# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [3]:
df["quality rating"].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [4]:
y = (df["quality rating"] >= 6).astype(int)
y.head()

0    1
1    1
2    0
3    1
4    0
Name: quality rating, dtype: int64

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [5]:
y.value_counts(normalize=True)

quality rating
0    0.5001
1    0.4999
Name: proportion, dtype: float64

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [6]:
X = df.drop(columns=["quality rating"])
X.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38


In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.612494,0.478552,0.479484,0.445078,0.399353,0.453899,0.477875,0.489762,0.520226,0.463207
std,0.102899,0.126472,0.123195,0.129879,0.111011,0.106351,0.117424,0.120383,0.109563,0.101430
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.559561,0.388628,0.392886,0.355014,0.324146,0.382479,0.399061,0.409364,0.445783,0.408541
50%,0.633229,0.495301,0.486661,0.434508,0.379824,0.454060,0.477700,0.489796,0.520263,0.439858
75%,0.674765,0.559211,0.569119,0.526649,0.461963,0.525641,0.557512,0.570228,0.593647,0.501779
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [8]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
results = {}

for solver in solvers:
    model = LogisticRegression(solver=solver, max_iter=1000)

    start = time.time()
    cv_results = cross_validate(model, X_scaled, y, cv=5, scoring="accuracy")
    elapsed = time.time() - start

    results[solver] = {
        "mean_fit_time": cv_results["fit_time"].mean(),
        "mean_accuracy": cv_results["test_score"].mean(),
        "total_time": elapsed
    }

results_df = pd.DataFrame(results).T.sort_values("mean_fit_time")
results_df

,mean_fit_time,mean_accuracy,total_time
liblinear,0.211800,0.86101,1.118659
newton-cg,0.244427,0.86123,1.312699
sag,0.345767,0.86124,1.764884
saga,0.650991,0.86124,3.290179
lbfgs,0.862856,0.86119,4.408123


In [9]:
fastest_solver = results_df["mean_fit_time"].idxmin()
print(f"En hızlı çözücü: {fastest_solver}")
print(f"Kesinlik puanları:\n{results_df['mean_accuracy']}")

En hızlı çözücü: liblinear
Kesinlik puanları:
liblinear    0.86101
newton-cg    0.86123
sag          0.86124
saga         0.86124
lbfgs        0.86119
Name: mean_accuracy, dtype: float64


<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [10]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-7.4.4, pluggy-1.4.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /home/arzu/S16D4-S-data-solvers/tests
plugins: dash-4.2.0, anyio-4.13.0
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.40s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [11]:
from sklearn.linear_model import SGDClassifier

sgd_model = SGDClassifier(loss="log_loss", max_iter=1000, random_state=42)

start = time.time()
sgd_cv_results = cross_validate(sgd_model, X_scaled, y, cv=5, scoring="accuracy")
sgd_elapsed = time.time() - start

print(f"SGD ortalama uyum süresi: {sgd_cv_results['fit_time'].mean():.4f} sn")
print(f"SGD ortalama kesinlik: {sgd_cv_results['test_score'].mean():.4f}")
print(f"\nKarşılaştırma için 2. bölüm sonuçları:\n{results_df}")

SGD ortalama uyum süresi: 0.1137 sn
SGD ortalama kesinlik: 0.8607

Karşılaştırma için 2. bölüm sonuçları:
           mean_fit_time  mean_accuracy  total_time
liblinear       0.211800        0.86101    1.118659
newton-cg       0.244427        0.86123    1.312699
sag             0.345767        0.86124    1.764884
saga            0.650991        0.86124    3.290179
lbfgs           0.862856        0.86119    4.408123


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [12]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [13]:
best_model = LogisticRegression(solver="lbfgs", max_iter=1000)
best_model.fit(X_scaled, y)

new_wine_scaled = pd.DataFrame(scaler.transform(new_wine), columns=new_wine.columns)

predicted_class = int(best_model.predict(new_wine_scaled)[0])
predicted_proba_of_class = best_model.predict_proba(new_wine_scaled)[0][predicted_class]

print(f"Tahmin edilen sınıf: {predicted_class}")
print(f"Bu sınıfa ait olasılık: {predicted_proba_of_class:.4f}")

Tahmin edilen sınıf: 0
Bu sınıfa ait olasılık: 0.9675


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [14]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-7.4.4, pluggy-1.4.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /home/arzu/S16D4-S-data-solvers/tests
plugins: dash-4.2.0, anyio-4.13.0
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.32s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master

